# ViInfographicVQA — OCR as a locator: zoom into the relevant region (single-image task)

The model sees each infographic downscaled to about one megapixel, while the originals are typically
2500 px wide or more, so small text becomes unreadable. Notebook 02 gives the model the OCR *text*, which is
noisy for Vietnamese (wrong diacritics and digits). This notebook uses OCR only to decide **where to look**:

1. run OCR once per image and keep the position of every text box;
2. match the question's words against the boxes (ignoring diacritics, tolerating small errors);
3. crop the best-matching regions from the full-resolution image;
4. give the model the whole image **plus** the sharp crops, and let it read them itself.

`CROP_MODE` selects the control experiments: `"random"` (same number and size of crops at random places) and
`"grid"` (2x2 tiling, no OCR). They show whether a gain comes from looking at the right place or just from more pixels.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LIMIT = None            # e.g. 50 for a quick trial run; None = the whole test split
SAVE_TO_DRIVE = True
LOAD_IN_4BIT = False    # set True for the 7B model on a 16 GB GPU

CROP_MODE = "locate"    # "locate" = OCR-guided crops | "random" = control | "grid" = 2x2 tiling, no OCR
N_CROPS = 2             # crops per question ("locate" and "random")
CROP_FRAC = 0.35        # each crop spans this share of the image width and height
USE_FORMAT_RULES = False  # True = also use the answer-format rules from notebook 03

OCR_CANVAS = 2560       # longest image side given to the OCR engine, in pixels
OCR_MIN_CONF = 0.3      # ignore OCR boxes below this confidence

# Whole image: same budget as the paper's Qwen wrapper. Each crop gets a smaller budget.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
CROP_MAX_PIXELS = 768 * 28 * 28
MAX_NEW_TOKENS = 100

DATASET = "duytranus/ViInfographicVQA"
SPLIT_FILE = "data/single_test.json"

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils Levenshtein bitsandbytes easyocr

In [ ]:
import json, os, random
from pathlib import Path

assert CROP_MODE in ("locate", "random", "grid")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/viinfographicvqa")
else:
    ROOT = Path("/content")
OUT_DIR = ROOT / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = MODEL_ID.split("/")[-1]
RUN_NAME = ("zoom-grid" if CROP_MODE == "grid" else f"zoom-{CROP_MODE}{N_CROPS}x{CROP_FRAC}") \
    + ("-format" if USE_FORMAT_RULES else "")
OUT_FILE = OUT_DIR / f"single_test__{MODEL_NAME}__{RUN_NAME}.json"
BOXES_FILE = ROOT / "ocr" / f"easyocr_canvas{OCR_CANVAS}_boxes.json"
BOXES_FILE.parent.mkdir(parents=True, exist_ok=True)
print("results ->", OUT_FILE)

## Data

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
from PIL import Image

DATA_DIR = Path("/content/ViInfographicVQA")
hf_hub_download(DATASET, SPLIT_FILE, repo_type="dataset", local_dir=DATA_DIR)
all_rows = json.loads((DATA_DIR / SPLIT_FILE).read_text(encoding="utf-8"))
rows = all_rows[:LIMIT] if LIMIT else all_rows
images = sorted({r["image_path"] for r in rows})
snapshot_download(DATASET, repo_type="dataset", local_dir=DATA_DIR, allow_patterns=images)
missing = [p for p in images if not (DATA_DIR / p).exists()]
assert not missing, f"{len(missing)} images failed to download, e.g. {missing[:3]}"
sizes = {p: Image.open(DATA_DIR / p).size for p in images}
widths = sorted(s[0] for s in sizes.values())
print(len(rows), "questions on", len(images), "images; median width", widths[len(widths) // 2], "px")

## OCR with box positions (once per image, cached; only needed for `CROP_MODE = "locate"`)

In [ ]:
import torch
from tqdm.auto import tqdm

boxes = {}
if CROP_MODE == "locate":
    boxes = json.loads(BOXES_FILE.read_text(encoding="utf-8")) if BOXES_FILE.exists() else {}
    todo = [p for p in images if p not in boxes]
    if todo:
        import easyocr
        reader = easyocr.Reader(["vi", "en"], gpu=torch.cuda.is_available(), verbose=False)
        for i, p in enumerate(tqdm(todo, desc="OCR")):
            dets = reader.readtext(str(DATA_DIR / p), canvas_size=OCR_CANVAS, min_size=10)
            boxes[p] = [[round(min(q[0] for q in b)), round(min(q[1] for q in b)),
                         round(max(q[0] for q in b)), round(max(q[1] for q in b)), text, round(float(conf), 3)]
                        for b, text, conf in dets]
            if i % 10 == 0:
                BOXES_FILE.write_text(json.dumps(boxes, ensure_ascii=False), encoding="utf-8")
        BOXES_FILE.write_text(json.dumps(boxes, ensure_ascii=False), encoding="utf-8")
        del reader
        torch.cuda.empty_cache()
    boxes = {p: [b for b in boxes[p] if b[5] >= OCR_MIN_CONF] for p in images}
    counts = sorted(len(v) for v in boxes.values())
    print(f"text boxes per image: median {counts[len(counts) // 2]}, min {counts[0]}, max {counts[-1]}")

## Choose the crops

The locator below is `viinfo/locate.py` from the repo. Word weights (IDF) come from the questions only.

In [ ]:
import math
import re
import unicodedata
from collections import Counter

import Levenshtein

TOKEN = re.compile(r"\d+(?:[.,]\d+)*|[a-z]+")
FUZZY_MIN = 0.8      # minimum similarity for an inexact word match
MIN_WINDOW_SCORE = 1.0  # windows scoring below this are not worth cropping


def fold(text: str) -> str:
    """Lowercase and strip Vietnamese diacritics: 'Bồ Đào Nha' -> 'bo dao nha'."""
    text = unicodedata.normalize("NFD", text.lower().replace("đ", "d"))
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")


def tokens(text: str) -> list[str]:
    return TOKEN.findall(fold(text))


def build_idf(questions: list[str]) -> dict[str, float]:
    """Inverse document frequency over the questions: 'bao nhiêu', 'theo', ... get ~0 weight."""
    df = Counter(t for q in questions for t in set(tokens(q)))
    n = len(questions)
    return {t: math.log((n + 1) / (c + 1)) for t, c in df.items()}


def _match(word: str, box_words: set[str]) -> float:
    if word in box_words:
        return 1.0
    if len(word) < 4 or word[0].isdigit():  # short words and numbers must match exactly
        return 0.0
    best = max((Levenshtein.ratio(word, w) for w in box_words), default=0.0)
    return best if best >= FUZZY_MIN else 0.0


def score_boxes(question: str, boxes: list, idf: dict[str, float]) -> list[float]:
    """Relevance of each OCR box to the question.

    A matched word counts more when it is rare among questions (not a stop word)
    and rare among this image's boxes (it pins down one place, unlike a word
    repeated all over the image). The sum is then scaled by how much of the box
    the question covers, so a short label fully named in the question
    ("HÀ NỘI") beats a long sentence that happens to share a few words.
    """
    default = max(idf.values(), default=1.0)  # unseen words are rare, hence informative
    box_words = [set(tokens(b[4])) for b in boxes]
    q_words = set(tokens(question))
    weight = {}
    for t in q_words:
        spread = sum(_match(t, words) > 0 for words in box_words)
        if spread:
            weight[t] = idf.get(t, default) / default * math.log(1 + len(boxes) / spread)
    scores = []
    for words in box_words:
        matched = sum(w * _match(t, words) for t, w in weight.items())
        covered = sum(_match(t, q_words) > 0 for t in words) / len(words) if words else 0.0
        scores.append(matched * (0.5 + covered))
    return scores


def _window(cx: float, cy: float, w: int, h: int, crop_frac: float) -> tuple[int, int, int, int]:
    cw, ch = w * crop_frac, h * crop_frac
    x0 = min(max(cx - cw / 2, 0), w - cw)
    y0 = min(max(cy - ch / 2, 0), h - ch)
    return round(x0), round(y0), round(x0 + cw), round(y0 + ch)


def _iou(a, b) -> float:
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


def boxes_in(window, boxes: list) -> list:
    """OCR boxes whose centre lies inside the window."""
    x0, y0, x1, y1 = window
    return [b for b in boxes if x0 <= (b[0] + b[2]) / 2 <= x1 and y0 <= (b[1] + b[3]) / 2 <= y1]


def locate(question: str, boxes: list, size: tuple[int, int], idf: dict[str, float],
           n_crops: int = 2, crop_frac: float = 0.35, max_iou: float = 0.3) -> list[tuple[int, int, int, int]]:
    """Up to `n_crops` windows (x0, y0, x1, y1) covering the text most related to the question.

    Each window spans `crop_frac` of the image width and height. A candidate
    window is centred on every matching box and scored by the matches it
    contains; windows are then picked greedily, skipping heavy overlaps.
    Returns [] when nothing in the image matches the question.
    """
    w, h = size
    scores = score_boxes(question, boxes, idf)
    hits = [(s, b) for s, b in zip(scores, boxes) if s > 0]
    candidates = []
    for _, anchor in hits:
        win = _window((anchor[0] + anchor[2]) / 2, (anchor[1] + anchor[3]) / 2, w, h, crop_frac)
        inside = sum(s for s, b in hits
                     if win[0] <= (b[0] + b[2]) / 2 <= win[2] and win[1] <= (b[1] + b[3]) / 2 <= win[3])
        candidates.append((inside, win))
    candidates.sort(key=lambda c: -c[0])
    chosen: list[tuple[int, int, int, int]] = []
    for total, win in candidates:
        if total < MIN_WINDOW_SCORE or len(chosen) == n_crops:
            break
        if all(_iou(win, c) <= max_iou for c in chosen):
            chosen.append(win)
    return chosen


def answer_coverage(answer: str, boxes: list) -> float:
    """Share of the gold answer's words that OCR found in `boxes` (diagnostic only, never used to answer)."""
    want = set(tokens(answer))
    if not want:
        return 0.0
    have = {t for b in boxes for t in tokens(b[4])}
    return sum(_match(t, have) > 0 for t in want) / len(want)


idf = build_idf([r["question"] for r in all_rows])


def grid_windows(size):
    w, h = size
    return [(x, y, x + w // 2, y + h // 2) for y in (0, h - h // 2) for x in (0, w - w // 2)]


def choose_windows(row):
    size = sizes[row["image_path"]]
    if CROP_MODE == "grid":
        return grid_windows(size)
    if CROP_MODE == "random":
        rng = random.Random(str(row["question_id"]))  # fixed per question, so reruns are identical
        return [_window(rng.uniform(0, size[0]), rng.uniform(0, size[1]), size[0], size[1], CROP_FRAC)
                for _ in range(N_CROPS)]
    return locate(row["question"], boxes[row["image_path"]], size, idf, n_crops=N_CROPS, crop_frac=CROP_FRAC)


windows = {r["question_id"]: choose_windows(r) for r in rows}
n = [len(w) for w in windows.values()]
print(f"crops per question: mean {sum(n) / len(n):.2f}; questions with no crop (answered from the whole image only): {n.count(0)}")

if boxes:
    # Diagnostic only: gold answers are used to measure the locator, never to answer.
    found = inside = 0
    for r in rows:
        b = boxes[r["image_path"]]
        if answer_coverage(r["answer"], b) >= 0.8:
            found += 1
            inside += answer_coverage(r["answer"], [x for w in windows[r["question_id"]] for x in boxes_in(w, b)]) >= 0.8
    print(f"gold answer readable by OCR somewhere in the image: {found}/{len(rows)}; "
          f"of those, inside the chosen crops: {inside / max(found, 1):.0%}")

In [ ]:
# Look at one example: the chosen crops drawn on the image.
from PIL import ImageDraw

example = next((r for r in rows if windows[r["question_id"]]), rows[0])
preview = Image.open(DATA_DIR / example["image_path"]).convert("RGB")
draw = ImageDraw.Draw(preview)
for win in windows[example["question_id"]]:
    draw.rectangle(win, outline=(255, 0, 0), width=max(4, preview.width // 200))
preview.thumbnail((700, 700))
print(example["question"], "->", example["answer"])
preview

## Model

In [ ]:
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
kwargs = dict(torch_dtype=torch.bfloat16, device_map="cuda")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, use_fast=True)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

## Inference

Same system prompt and greedy decoding as notebook 01. The user message holds the whole image, then the crops.

In [ ]:
from functools import lru_cache

PAPER_PROMPT = ("Answer the following question based solely on the image content "
                "concisely with a single term.")
FORMAT_PROMPT = """Answer the following question based solely on the image content. Give only the answer, with no explanation, following these rules:
- If the question asks you to count or calculate (how many, total, difference), answer with the number only, without a unit or extra words.
- If the answer is a value shown in the image, copy it exactly as written there, including its unit or symbol (for example % or °C).
- If the answer is text shown in the image, copy the words exactly as they appear. Do not paraphrase or translate them.
- If the question offers options, answer with the chosen option copied exactly from the question.
- If several items are asked for, list all of them, separated by commas."""
SYSTEM_PROMPT = FORMAT_PROMPT if USE_FORMAT_RULES else PAPER_PROMPT
CROP_NOTE = "The images after the first one are enlarged crops of regions of the first image that may be relevant."


@lru_cache(maxsize=2)
def load_image(path: str):
    return Image.open(path).convert("RGB")


def build_messages(question: str, image_path: str, wins: list) -> list:
    full = load_image(image_path)
    content = [{"type": "image", "image": full, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS}]
    for win in wins:
        content.append({"type": "image", "image": full.crop(win),
                        "min_pixels": MIN_PIXELS, "max_pixels": CROP_MAX_PIXELS})
    note = f"{CROP_NOTE}\n" if wins else ""
    content.append({"type": "text", "text": f"{note}Question: {question}\nAnswer:"})
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": content}]


def parse_answer(response: str) -> str:
    return response.split("Answer:")[-1].strip() if "Answer:" in response else response.strip()


@torch.no_grad()
def infer(question: str, image_path: str, wins: list) -> str:
    messages = build_messages(question, image_path, wins)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return parse_answer(processor.batch_decode(new_tokens, skip_special_tokens=True)[0])


done = {}
if OUT_FILE.exists():
    done = {r["question_id"]: r["predict"] for r in json.loads(OUT_FILE.read_text(encoding="utf-8"))
            if "predict" in r}
    print("resuming:", len(done), "answers already saved")


def save():
    OUT_FILE.write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")


for i, row in enumerate(tqdm(rows)):
    row["crops"] = [list(w) for w in windows[row["question_id"]]]
    if row["question_id"] in done:
        row["predict"] = done[row["question_id"]]
        continue
    row["predict"] = infer(row["question"], str(DATA_DIR / row["image_path"]), windows[row["question_id"]])
    if i % 25 == 0:
        save()
save()
print("saved", OUT_FILE)

## Score

`ANLS` reproduces the paper's scorer exactly; `ANLS-std` is the textbook definition; `Acc` is exact match.
Paper, single-image overall ANLS: Qwen2.5-VL-7B 67.42 · fine-tuned 67.80 · InternVL3.5-8B 67.02 · Ovis2.5-9B 71.02.

In [ ]:
from collections import defaultdict

import Levenshtein

THRESHOLD = 0.5


def clean_prediction(text: str) -> str:
    return text.lower().strip().rstrip(".").replace('"', "").rstrip(">").lstrip("<")


def clean_answer(text: str) -> str:
    return text.lower().strip()


def paper_anls(answer: str, prediction: str) -> float:
    p = clean_prediction(prediction).replace('"', "").rstrip(".").lower()
    score = Levenshtein.ratio(p, clean_answer(answer))
    return score if score >= THRESHOLD else 0.0


def standard_anls(answer: str, prediction: str) -> float:
    p, g = clean_prediction(prediction), clean_answer(answer)
    if not p and not g:
        return 1.0
    nl = Levenshtein.distance(p, g) / max(len(p), len(g))
    return 1.0 - nl if nl < THRESHOLD else 0.0


def exact_match(answer: str, prediction: str) -> float:
    return float(clean_prediction(prediction) == clean_answer(answer))


METRICS = {"anls": paper_anls, "anls_std": standard_anls, "acc": exact_match}

# Fields to break results down by; whichever exist in the split are used.
GROUP_FIELDS = ("answer_source", "element", "image_type")


def _as_labels(value) -> list[str]:
    """`element` is a string in most rows but a list in a few."""
    if value is None:
        return []
    return [str(v) for v in value] if isinstance(value, list) else [str(value)]


def score(rows: list[dict], pred_key: str = "predict") -> dict:
    """Return {"overall": {...}, "<field>": {"<label>": {...}}} with scores in percent."""
    sums: dict = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    for row in rows:
        vals = {name: fn(row["answer"], str(row.get(pred_key, ""))) for name, fn in METRICS.items()}
        groups = [("overall", "overall")]
        groups += [(f, label) for f in GROUP_FIELDS for label in _as_labels(row.get(f))]
        for field, label in groups:
            bucket = sums[field][label]
            bucket["count"] += 1
            for name, v in vals.items():
                bucket[name] += v
    out: dict = {}
    for field, labels in sums.items():
        table = {label: {"count": int(b["count"]),
                         **{m: round(100 * b[m] / b["count"], 2) for m in METRICS}}
                 for label, b in labels.items()}
        out[field] = table["overall"] if field == "overall" else table
    return out


def format_report(result: dict) -> str:
    lines = [f"{'':32s}{'n':>6s}{'ANLS':>8s}{'ANLS-std':>10s}{'Acc':>8s}"]

    def row(name: str, s: dict) -> str:
        return f"{name[:31]:32s}{s['count']:6d}{s['anls']:8.2f}{s['anls_std']:10.2f}{s['acc']:8.2f}"

    lines.append(row("Overall", result["overall"]))
    for field in GROUP_FIELDS:
        if field in result:
            lines.append(f"-- {field}")
            for label, s in sorted(result[field].items(), key=lambda kv: -kv[1]["count"]):
                lines.append(row("  " + label, s))
    return "\n".join(lines)


print(MODEL_ID, "|", RUN_NAME)
print(format_report(score(rows)))

## Compare every run saved for this model (ANLS, on the questions all runs answered)

In [ ]:
runs = {}
for path in sorted(OUT_DIR.glob(f"single_test__{MODEL_NAME}*.json")):
    name = path.stem.split("__")[2] if path.stem.count("__") >= 2 else "baseline"
    runs[name] = {r["question_id"]: r for r in json.loads(path.read_text(encoding="utf-8")) if "predict" in r}
common = set.intersection(*(set(v) for v in runs.values()))
names = sorted(runs, key=lambda n: (n != "baseline", n))
scores = {n: score([runs[n][q] for q in sorted(common)]) for n in names}
print(f"{len(common)} questions answered by all {len(names)} runs\n")
print(f"{'':28s}{'n':>6s}" + "".join(f"{n[:27]:>29s}" for n in names))


def line(label, pick):
    cells = [pick(scores[n]) for n in names]
    print(f"{label[:27]:28s}{cells[0]['count']:6d}" + "".join(f"{c['anls']:29.2f}" for c in cells))


line("Overall", lambda s: s["overall"])
for field in ("answer_source", "element"):
    print("--", field)
    ref = scores[names[0]][field]
    for label in sorted(ref, key=lambda k: -ref[k]["count"]):
        line("  " + label, lambda s, f=field, l=label: s[f][l])

if "baseline" in runs and RUN_NAME in runs:
    def ok(run, q):
        return paper_anls(runs[run][q]["answer"], runs[run][q]["predict"]) > 0
    fixed = [q for q in sorted(common) if not ok("baseline", q) and ok(RUN_NAME, q)]
    broken = [q for q in sorted(common) if ok("baseline", q) and not ok(RUN_NAME, q)]
    print(f"\n{RUN_NAME} vs baseline -> wrong to right: {len(fixed)}   right to wrong: {len(broken)}")
    for title, group in (("FIXED", fixed), ("BROKEN", broken)):
        print(f"\n== {title} (first 8)")
        for q in group[:8]:
            r = runs[RUN_NAME][q]
            print(f"[{r['answer_source']} | {r['element']}] {r['question']}\n   gold: {r['answer']}\n"
                  f"   base: {runs['baseline'][q]['predict']}\n   new : {r['predict']}\n")
else:
    print("\nNo baseline run found: run notebook 01 with the same MODEL_ID to get the fixed/broken lists.")